# Complementos de Investigação Operacional — capítulo 3: Problemas sequenciais

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/ex03_1_johnson.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex03_1_johnson.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.1 Problemas sequenciais: regra de Johnson (2 máquinas)

Exemplo [`ex03_1_johnson.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/ex03_1_johnson.py) · funções da UC: [`sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/sequenciamento.py)

```text
Reproduz os exemplos do deck 3.1 (regra de Johnson, 2 máquinas).

Gráfica, seis trabalhos (impressão, acabamento): pela ordem de chegada T = 53 h (ociosos 12 e 11);
Johnson D, E, A, F, B, C com T = 45 h (ociosos 4 e 3).
Força bruta: o mínimo das 720 sequências é 45 h (só D E A F B C e D E F A B C), média 53 h, pior 61 h.
Para resolver na aula: Exemplo 1 (nove trabalhos) T = 61, ociosos 11 e 2;
Exemplo 2 (sete trabalhos) T = 67, ociosos 1 e 17, e os empates não mudam o tempo total.
```

In [2]:
from sequenciamento import tempos, cmax, ocios, tempo_total, johnson, forca_bruta


def mostra(nome, seq, P):
    """Sequência, tempo total, ociosos e a tabela dos instantes de conclusão."""
    C = tempos(seq, P)
    print("  %s: %s  T = %d  ociosos = %s" % (nome, " ".join(seq), cmax(seq, P),
                                             ", ".join("%d" % x for x in ocios(seq, P))))
    for i in range(len(C[0])):
        print("    M%d:%s" % (i + 1, "".join("%4d" % C[k][i] for k in range(len(seq)))))


ok = True
print("Complementos de IO — deck 3.1: regra de Johnson (2 máquinas)")

# ------------------------------------------------------------ a gráfica (slide «Em Python»)
a = dict(A=8, B=11, C=5, D=3, E=4, F=10)
b = dict(A=9, B=5, C=2, D=5, E=12, F=9)
P = {j: (a[j], b[j]) for j in a}
print("\nGráfica (impressão, acabamento): somas %d e %d h" % (sum(a.values()), sum(b.values())))
mostra("ordem de chegada", list(P), P)
s = johnson(a, b)
mostra("Johnson", s, P)
print("  tempo_total(Johnson) = %d" % tempo_total(s, a, b))
ok &= cmax(list(P), P) == 53 and ocios(list(P), P) == [12, 11]
ok &= s == list("DEAFBC") and tempo_total(s, a, b) == 45 and ocios(s, P) == [4, 3]
ok &= [r[1] for r in tempos(s, P)] == [8, 20, 29, 38, 43, 45]

# ------------------------------------------------------------ força bruta: as 720 sequências
bf = forca_bruta(P)
vals = [v for v, _ in bf]
otimas = [q for v, q in bf if v == vals[0]]
print("\nForça bruta: %d sequências; mínimo %d (%s); média %.2f; pior %d"
      % (len(bf), vals[0], ", ".join(otimas), sum(vals) / len(vals), max(vals)))
hist = [(T, vals.count(T)) for T in range(min(vals), max(vals) + 1) if T in vals]
print("  n.º de sequências por T: " + ", ".join("%d: %d" % h for h in hist))
ok &= vals[0] == 45 and otimas == ["DEAFBC", "DEFABC"] and max(vals) == 61
ok &= round(sum(vals) / len(vals)) == 53 and hist[1] == (46, 38) and hist[5] == (50, 123)

# ------------------------------------------------------------ Exemplo 1: nove trabalhos
M1 = [2, 5, 4, 9, 6, 8, 7, 5, 4]
M2 = [6, 8, 7, 4, 3, 9, 3, 8, 11]
nomes = ["J%d" % k for k in range(1, 10)]
a1 = dict(zip(nomes, M1)); b1 = dict(zip(nomes, M2))
P1 = {j: (a1[j], b1[j]) for j in nomes}
print("\nExemplo 1 (nove trabalhos): somas %d e %d" % (sum(M1), sum(M2)))
s1 = johnson(a1, b1)
mostra("Johnson", s1, P1)
alt = "J1 J9 J3 J8 J2 J6 J4 J7 J5".split()
print("  com os empates escolhidos ao contrário: %s  T = %d" % (" ".join(alt), cmax(alt, P1)))
ok &= s1 == "J1 J3 J9 J2 J8 J6 J4 J5 J7".split() and cmax(s1, P1) == 61 and ocios(s1, P1) == [11, 2]
ok &= cmax(alt, P1) == 61

# ------------------------------------------------------------ Exemplo 2: sete trabalhos
M1 = [3, 12, 15, 6, 10, 11, 9]
M2 = [8, 10, 10, 6, 12, 1, 3]
nomes = [str(k) for k in range(1, 8)]
a2 = dict(zip(nomes, M1)); b2 = dict(zip(nomes, M2))
P2 = {j: (a2[j], b2[j]) for j in nomes}
print("\nExemplo 2 (sete trabalhos): somas %d e %d" % (sum(M1), sum(M2)))
s2 = johnson(a2, b2)
mostra("Johnson", s2, P2)
print("  empates (o 4 tem a = b = 6; o 2 e o 3 têm b = 10): outras sequências de Johnson")
variantes = ["1 5 4 2 3 7 6", "1 5 3 2 4 7 6", "1 5 2 3 4 7 6", "1 4 5 3 2 7 6"]
for v in variantes:
    print("    %s  T = %d" % (v, cmax(v.split(), P2)))
ok &= s2 == "1 4 5 2 3 7 6".split() and cmax(s2, P2) == 67 and ocios(s2, P2) == [1, 17]
ok &= all(cmax(v.split(), P2) == 67 for v in variantes)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 3.1: regra de Johnson (2 máquinas)

Gráfica (impressão, acabamento): somas 41 e 42 h
  ordem de chegada: A B C D E F  T = 53  ociosos = 12, 11
    M1:   8  19  24  27  31  41
    M2:  17  24  26  32  44  53
  Johnson: D E A F B C  T = 45  ociosos = 4, 3
    M1:   3   7  15  25  36  41
    M2:   8  20  29  38  43  45
  tempo_total(Johnson) = 45

Força bruta: 720 sequências; mínimo 45 (DEAFBC, DEFABC); média 52.99; pior 61
  n.º de sequências por T: 45: 2, 46: 38, 47: 15, 48: 17, 49: 17, 50: 123, 51: 41, 52: 96, 53: 76, 54: 58, 55: 51, 56: 46, 57: 30, 58: 58, 59: 20, 60: 20, 61: 12

Exemplo 1 (nove trabalhos): somas 50 e 59
  Johnson: J1 J3 J9 J2 J8 J6 J4 J5 J7  T = 61  ociosos = 11, 2
    M1:   2   6  10  15  20  28  37  43  50
    M2:   8  15  26  34  42  51  55  58  61
  com os empates escolhidos ao contrário: J1 J9 J3 J8 J2 J6 J4 J7 J5  T = 61

Exemplo 2 (sete trabalhos): somas 66 e 50
  Johnson: 1 4 5 2 3 7 6  T = 67  ociosos = 1, 17
    M1:   3   9  19  31